<a href="https://colab.research.google.com/github/novay/thesis/blob/main/1.%20base-models/gemma3_4bit_wikimedia).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inisialisasi**

- Base models: gemma-3-1b-it
- Dataset: wikimedia/wikipedia

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model_name = "gemma-3-1b-it-unsloth-bnb-4bit"
base_model = f"unsloth/{model_name}"
model_path = f"/content/drive/MyDrive/Thesis/Assets/Phase_1/{model_name}"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **1. Preparasi Dataset**

*Hanya dilakukan sekali pada saat tuning model awal, agar dataset yang digunakan untuk training dan evaluasi sama.

https://huggingface.co/datasets/wikimedia/wikipedia/viewer/20231101.id

### 1.1. Train

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikimedia/wikipedia", "20231101.id", split = "train",)

# We select 10% of the data to make training faster!
dataset = dataset.train_test_split(train_size = 0.1)["train"]
dataset.save_to_disk("/content/drive/MyDrive/Thesis/Assets/Wikipedia/train_dataset")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/146M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/170M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/665622 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/66562 [00:00<?, ? examples/s]

### 1.2. Evaluasi

In [ ]:
from datasets import load_dataset
from datasets import load_from_disk

eval_dataset = load_dataset("wikimedia/wikipedia", "20231101.id", split="train[1%:2%]")
eval_dataset.save_to_disk("/content/drive/MyDrive/Thesis/Assets/Wikipedia/eval_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/6656 [00:00<?, ? examples/s]

### 1.2. Load from Disk

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("/content/drive/MyDrive/Thesis/Assets/Wikipedia/train_dataset")
eval_dataset = load_from_disk("/content/drive/MyDrive/Thesis/Assets/Wikipedia/eval_dataset")

# **2. Language Fine-tuning**

### 2.1. Inisiasi Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True
)

==((====))==  Unsloth 2025.7.3: Fast Gemma3 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head"], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,   # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:574: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['model.embed_tokens', 'lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(


Unsloth: Making `model.base_model.model.model.embed_tokens` require gradients


In [ ]:
%%time

wikipedia_prompt = """Artikel Wikipedia
### Judul: {}

### Artikel:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    titles = examples["title"]
    texts  = examples["text"]
    outputs = []
    for title, text in zip(titles, texts):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = wikipedia_prompt.format(title, text) + EOS_TOKEN
        outputs.append(text)
    return { "text" : outputs, }
pass

dataset = dataset.map(formatting_prompts_func, batched = True)

CPU times: user 31.8 ms, sys: 5.14 ms, total: 36.9 ms
Wall time: 5.07 s


<a name="Train"></a>
### 2.2. Fine-tuning

Custom Parameter:
- Step 120
- Epoch 2

In [ ]:
from transformers import TrainingArguments
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model, # Tentukan model yang akan di-finetune
    tokenizer = tokenizer, # untuk memproses teks input menjadi token
    train_dataset = dataset, # Dataset yang digunakan untuk pelatihan
    dataset_text_field = "text", # Nama kolom dalam train_dataset yang berisi teks untuk pelatihan
    max_seq_length = max_seq_length, # Menentukan panjang maksimum input yang akan ditokenisasi
    dataset_num_proc = 2, # Mengatur jumlah CPU yang digunakan untuk memproses dataset

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2, # Jumlah sampel per perangkat (GPU/CPU) dalam satu batch pelatihan
        gradient_accumulation_steps = 8, # Mengakumulasi gradien dari beberapa batch kecil sebelum memperbarui model, mensimulasikan batch size yang lebih besar tanpa meningkatkan penggunaan memori.

        # Use warmup_ratio and num_train_epochs for longer runs!
        max_steps = 120,
        warmup_steps = 10,
        # warmup_ratio = 0.1,
        num_train_epochs = 2,

        # Select a 2 to 10x smaller learning rate for the embedding matrices!
        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5,

        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = model_path,
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
%%time

trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66,562 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 138,067,968 of 1,171,655,808 (11.78% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.558300
2,3.908700
3,3.513900
4,2.612500
5,3.349400
6,3.412600
7,2.639100
8,2.822600
9,2.969100
10,2.820000


/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:222: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


CPU times: user 10min 39s, sys: 16.4 s, total: 10min 56s
Wall time: 11min 22s


In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
2.943 GB of memory reserved.


### 2.3. Log Trainer

In [ ]:
import json
import math

def print_trainer_stats_json(trainer_stats):
    stats_dict = {
        "Global Steps": trainer_stats.global_step,
        "Training Loss": round(trainer_stats.training_loss, 4),
        "Perplexity": round(math.exp(trainer_stats.training_loss), 2),
        "Metrics": {
            "Runtime (s)": round(trainer_stats.metrics['train_runtime'], 2),
            "Samples per Second": round(trainer_stats.metrics['train_samples_per_second'], 3),
            "Steps per Second": round(trainer_stats.metrics['train_steps_per_second'], 3),
            "Total FLOPs": f"{trainer_stats.metrics['total_flos']:.2e}"
        }
    }
    print("Training Statistics:")
    print(json.dumps(stats_dict, indent=2))

print_trainer_stats_json(trainer_stats)

Training Statistics:
{
  "Global Steps": 120,
  "Training Loss": 2.4868,
  "Perplexity": 12.02,
  "Metrics": {
    "Runtime (s)": 680.06,
    "Samples per Second": 2.823,
    "Steps per Second": 0.176,
    "Total FLOPs": "5.59e+15"
  }
}


# **3. Evaluasi**

### 3.1. TTFT (Time to First Token)

In [ ]:
%%time

from unsloth import FastLanguageModel
import torch
import time
import os
import gc

# Set environment variable untuk mengurangi fragmentasi memori
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Bersihkan memori GPU
torch.cuda.empty_cache()
gc.collect()

# Muat model dan tokenizer (lanjutan dari kode Anda)
# Changed model_name to load from the checkpoint path
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{model_path}/checkpoint-120", # Load from the checkpoint path
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True
)

FastLanguageModel.for_inference(model)
# model.to("cuda")  # Pastikan model di GPU - This line is removed as it's not needed for 8-bit models

# Format prompt seperti pada finetuning
wikipedia_prompt = """Artikel Wikipedia
### Judul: {}
### Artikel:
{}"""
EOS_TOKEN = tokenizer.eos_token

# Daftar prompt untuk mengukur TTFT
prompts = [
    "Jelaskan secara singkat sejarah Indonesia.",
    "Apa yang dimaksud dengan Pancasila menurut artikel Wikipedia?",
    "Berikan ringkasan tentang budaya Jawa.",
    "Deskripsikan sistem pemerintahan Kesultanan Mataram.",
    "Apa itu Gamelan dan bagaimana perannya dalam budaya Indonesia?",
    "Jelaskan perkembangan ekonomi Indonesia pasca-kemerdekaan."
]

# Fungsi untuk menghitung TTFT
def calculate_ttft(prompt_text, model, tokenizer, max_seq_length=2048):
    formatted_prompt = wikipedia_prompt.format("Tes", prompt_text) + EOS_TOKEN
    inputs = tokenizer(
        [formatted_prompt],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_seq_length
    ).to("cuda")

    start_time = time.time()
    outputs = model.generate(**inputs, max_new_tokens=1)  # Hanya 1 token untuk TTFT
    ttft = time.time() - start_time

    # Bersihkan memori
    torch.cuda.empty_cache()
    return ttft

# Hitung TTFT untuk setiap prompt
results = []
for prompt_text in prompts:
    ttft = calculate_ttft(prompt_text, model, tokenizer)
    results.append({"prompt": prompt_text, "ttft": ttft})

    # Cetak hasil per prompt
    print(f"Prompt: {prompt_text}")
    print(f"TTFT: {ttft:.4f} seconds\n")

# Cetak rata-rata TTFT
avg_ttft = sum([res["ttft"] for res in results]) / len(results)
print(f"Model: {model_name}")
print(f"Average TTFT: {avg_ttft:.4f} seconds")
print(f"=======================================")
# Bersihkan memori akhir
torch.cuda.empty_cache()
gc.collect()

==((====))==  Unsloth 2025.7.3: Fast Gemma3 patching. Transformers: 4.53.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Prompt: Jelaskan secara singkat sejarah Indonesia.
TTFT: 7.3838 seconds

Prompt: Apa yang dimaksud dengan Pancasila menurut artikel Wikipedia?
TTFT: 0.2252 seconds

Prompt: Berikan ringkasan tentang budaya Jawa.
TTFT: 0.1370 seconds

Prompt: Deskripsikan sistem pemerintahan Kesultanan Mataram.
TTFT: 0.1376 seconds

Prompt: Apa itu Gamelan dan bagaimana perannya dalam budaya Indonesia?
TTFT: 0.1372 seconds

Prompt: Jelaskan perkembangan ekonomi Indonesia pasca-kemerdekaan.
TTFT: 0.1374 seconds

Model: gemma-3-1b-it-unsloth-bnb

6602

### 3.2. Perplexity

In [ ]:
%%time

from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

# Format prompt seperti pada finetuning
wikipedia_prompt = """Artikel Wikipedia
### Judul: {}
### Artikel:
{}"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    titles = examples["title"]
    texts = examples["text"]
    outputs = [wikipedia_prompt.format(title, text) + EOS_TOKEN for title, text in zip(titles, texts)]
    return {"text": outputs}


# Set environment variable untuk mengurangi fragmentasi memori
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Bersihkan memori GPU
torch.cuda.empty_cache()
gc.collect()

# Konfigurasi
max_seq_length = 1024  # Dikurangi untuk menghemat memori
dtype = None
load_in_4bit = True
batch_size = 1  # Batch size kecil untuk mencegah out-of-memory


# model.to("cuda")  # Pastikan model di GPU - This line is removed as it's not needed for 8-bit models

# Terapkan formatting dan hapus semua kolom kecuali 'text'
eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True, remove_columns=eval_dataset.column_names)

# Simpan dataset evaluasi untuk konsistensi
eval_dataset.save_to_disk("./eval_wiki_id")

# Tokenisasi dataset evaluasi
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
        return_tensors="pt"
    )

tokenized_eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]  # Pastikan hanya input_ids, attention_mask, dll yang tersisa
)
tokenized_eval_dataset.save_to_disk("./tokenized_eval_wiki_id")

# Buat DataLoader
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
eval_dataloader = DataLoader(tokenized_eval_dataset, batch_size=batch_size, collate_fn=data_collator)

# Hitung Perplexity
model.eval()
total_loss = 0
total_tokens = 0

with torch.no_grad():
    for batch in eval_dataloader:
        inputs = {key: val.to("cuda") for key, val in batch.items() if key in ["input_ids", "attention_mask", "labels"]}
        try:
            outputs = model(**inputs)
            loss = outputs.loss
            total_loss += loss.item() * inputs["input_ids"].size(1)
            total_tokens += inputs["input_ids"].size(1)
        except RuntimeError as e:
            print(f"Error during batch processing: {e}")
            torch.cuda.empty_cache()
            continue
        # Bersihkan memori setelah setiap batch
        torch.cuda.empty_cache()

if total_tokens == 0:
    raise ValueError("No valid tokens processed. Check dataset or model compatibility.")

avg_loss = total_loss / total_tokens
perplexity = torch.exp(torch.tensor(avg_loss)).item()

# Cetak hasil
print(f"Model: {model_name}")
print(f"Perplexity: {perplexity:.2f}")
print(f"=======================================")

# Bersihkan memori akhir
torch.cuda.empty_cache()
gc.collect()

Saving the dataset (0/1 shards):   0%|          | 0/6656 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6656 [00:00<?, ? examples/s]

Model: gemma-3-1b-it-unsloth-bnb-4bit
Perplexity: 54.00
CPU times: user 14min 8s, sys: 19.2 s, total: 14min 27s
Wall time: 14min 30s


42577